In [1]:
import json
import pickle
import pandas as pd
from tqdm.notebook import tqdm
from rca import linear_probe

In [2]:
with open('../../data/brain_behav_union.pkl', 'rb') as f:
    brain_behav_union = pickle.load(f)

# Loading dictionary of dtype to embed
with open('../../data/dtype_to_embed.json', 'r') as f:
    dtype_to_embed = json.load(f)

with open('../../data/embed_to_dtype.json', 'r') as f:
    embed_to_dtype = json.load(f)

brain_behav_names = dtype_to_embed['brain'] + dtype_to_embed['behavior']

Llama_3_8B = pd.read_csv('../../data/embeds/Llama_3_8B.csv', index_col=0)

# Subsetting to brain and behavior vocab
Llama_3_8B = Llama_3_8B.loc[Llama_3_8B.index.intersection(brain_behav_union)]

# Standardising
Llama_3_8B = (Llama_3_8B - Llama_3_8B.mean()) / Llama_3_8B.std()

Llama_3_8B

,0,1,2,3,4,5,6,7,8,9,...,4086,4087,4088,4089,4090,4091,4092,4093,4094,4095
ABC,-0.121873,-0.372243,0.294516,-0.966298,0.810673,-1.890804,-1.173450,-0.921663,-0.098159,2.064089,...,-0.815725,-1.352378,0.917316,-1.428172,-0.566973,0.413507,-2.203377,-2.218246,1.734215,-0.937996
AI,-1.504516,1.031783,-0.709001,-3.365720,1.952616,0.118293,0.296595,0.013925,1.047892,0.462469,...,1.837624,-1.574351,1.778785,0.856182,-0.498651,0.225083,-1.261701,0.326375,-2.183356,0.195282
Aaron,-0.433314,0.788519,-3.867654,1.877590,0.715968,-0.485366,-0.429883,0.459521,-1.464101,-0.295625,...,-3.197491,3.384878,1.796307,2.147621,-0.183775,0.336141,0.780301,-0.940400,-3.775926,-0.414030
Abe,-2.104328,2.250219,-2.121075,2.766522,-2.790840,-0.414196,-0.794027,2.264926,1.099390,3.500024,...,-2.051178,2.676417,2.070809,0.964884,-2.869133,-2.746023,-2.872103,-0.997334,-3.340223,0.369371
Abel,-2.578026,0.671061,0.580506,0.296133,-2.137517,-0.294008,-1.606348,-0.898643,0.623232,-1.052857,...,-1.917886,-0.838143,2.751224,0.472280,0.601188,-0.587260,-0.787452,2.107135,-4.872696,-1.573209
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
zoom,-0.703998,0.244987,0.994169,1.464376,1.680855,0.396880,0.154531,0.013925,1.403478,0.514245,...,0.631329,-1.352378,0.834089,1.888036,-2.037385,-0.213532,-1.865602,-1.807058,-0.134191,-1.934975
zooming,-0.391788,0.585078,0.937993,0.781183,1.241646,0.384056,-0.246537,-0.948793,2.322281,0.007699,...,0.413063,-0.397898,0.213539,1.196889,-1.746273,0.445951,-1.230995,-0.908770,0.475934,-2.340900
zoophobia,0.310299,-0.894279,0.138754,-0.524219,-0.068603,1.458652,-0.888246,0.405260,0.389853,-0.321945,...,-0.191751,1.925411,-2.102208,-1.067996,-1.270989,-0.477450,2.607355,0.925761,0.266534,-1.151998
zucchini,-0.001910,0.092983,0.309837,0.483425,0.518324,1.407359,1.043246,-1.197900,-0.430857,-0.845751,...,-2.204464,0.138535,1.991962,-0.482306,-0.519445,-0.791906,-0.112755,1.159031,-0.014231,-1.834767


In [ ]:
# Loading norms
norms = pd.read_csv('../../data/psychNorms/psychNorms_processed.zip', index_col=0, compression='zip', low_memory=False)
norm_meta = pd.read_csv('../../data/psychNorms/psychNorms_metadata_processed.csv', index_col='norm')
norms

In [ ]:
results = []
for norm_name in tqdm(norms.columns()):
    norms_results = linear_probe('Llama_3_8B', Llama_3_8B, norm_name, norms, norm_meta, embed_to_dtype, n_jobs=-1)
    results.append(norms_results)


results = pd.DataFrame(
    results,
    columns=['embed', 'embed_type', 'norm', 'train_n', 'test_n', 'p', 'r2_mean', 'r2_sd', 'check']
)
results

In [ ]:
results.to_csv('../../data/results/rca_llama.csv', index=False)
results